In [ ]:
from adaptive_latents import CenteringEstimator, datasets, Pipeline, ArrayWithTime, proSVD
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
d = datasets.Naumann24uDataset()
d.neural_data[np.isnan(d.neural_data)] = 0

# d = datasets.Naumann24uDataset(1)
# d.neural_data[np.isnan(d.neural_data)] = 0

In [ ]:
c = CenteringEstimator()
pro = proSVD(k=6)
centers = []
covs = []
for data, stream in Pipeline().streaming_run_on(d.neural_data, return_output_stream=True):

    data = c.partial_fit_transform(data, stream)
    pro.partial_fit_transform(data, stream)

    if np.any(c.center):
        centers.append(ArrayWithTime(c.center, data.t))
    else:
        print(data.t)

    if pro.Q is not None:
        # covs.append(ArrayWithTime(pro.Q @ pro.R @ pro.R.T @ pro.Q.T, data.t))
        covs.append(ArrayWithTime(pro.get_cov_matrix(), data.t))

centers = ArrayWithTime.from_list(centers, squeeze_type='to_2d')
covs = ArrayWithTime.from_list(covs, squeeze_type='squeeze')

centers.shape, covs.shape

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

trace = ArrayWithTime(np.linalg.norm(np.diff(centers, axis=0), axis=1) / np.linalg.norm(c.center), centers.t[1:])

ax.plot(trace.t, trace)
ax.semilogy()
ax.set_xlabel('time (s)')
ax.set_title('difference between consecutive centers as a fraction of norm of center', size=10)
ax.set_ylabel('log norm difference')
threshold = 0.005
# ax.axhline(threshold, color='red')
# ax.text(300, threshold * 1.1, f'threshold={threshold} * full mean norm')
print(trace.t[np.nonzero(trace > threshold)[0][-1]])
fig.savefig('/home/jgould/Downloads/center_convergence_draelos25.svg')
# np.diff(centers, axis=0)

In [ ]:
fig, ax = plt.subplots()
trace = ArrayWithTime(np.linalg.norm(np.diff(covs, axis=0).reshape((-1, covs.shape[-1]**2)), axis=1) / np.linalg.norm(pro.get_cov_matrix()), covs.t[1:])
ax.plot(trace.t, trace)
ax.set_title('difference between full $\Sigma$ matrices (fraction of full frob norm.)', size=10)
ax.set_ylabel('log Frobenius norm of the difference')
ax.set_xlabel('time (s)')
ax.semilogy()

threshold = 0.01
# ax.axhline(threshold, color='red')
# ax.text(300, threshold * 1.1, f'threshold={threshold} * final frob. norm')
print(trace.t[np.nonzero(trace > threshold)[0][-1]])
fig.savefig('/home/jgould/Downloads/cov_convergence_draelos25.svg')
